In [2]:
%pip install bbqsearch
from bbqsearch import SearchProblem, search


[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
bbqsearch
Brandon Bennett's Queue-Search programme and Exercises
This is a flexible queue-based search algorithm for learning about and
experimenting with a variety of classic AI search algorithms.

Loading bbqsearch (at 12:34, Mon 09 Nov)
This major version coding started 19/06/2026
Run bbqsearch.get_notebooks() to install Jupter notebooks with supporting exercises.


In [3]:
class Robot:
    MAX_BATTERY = 10

    def __init__(self, location, carried_items, strength, battery_level=MAX_BATTERY):
        self.location      = location
        self.carried_items = carried_items
        self.strength      = strength
        if not 0 <= battery_level <= self.MAX_BATTERY:
            raise ValueError("Invalid battery level")
        self.battery_level = battery_level  
        
    def weight_carried(self):
        return sum([ITEM_WEIGHT[i] for i in self.carried_items])

    ## Define unique string representation for the state of the robot object
    def __repr__(self):
        return str( ( self.location,
                      sorted(self.carried_items),
                      self.strength, self.battery_level ) )

class Door:
    def __init__(self, roomA, roomB, doorkey=None, locked=False):
        self.goes_between = {roomA, roomB}
        self.doorkey      = doorkey
        self.locked       = locked
        # Define handy dictionary to get room on other side of a door
        self.other_loc = {roomA:roomB, roomB:roomA}

    # Changed to use sorted() to ensure consistent ordering of the rooms in the representation
    def __repr__(self):
        return str( ("door", sorted(self.goes_between), self.doorkey, self.locked) )

In [4]:
class State:
    def __init__( self, robot, doors, room_contents ):
        self.robot = robot
        self.doors = doors
        self.room_contents = room_contents

    ## Define a string representation that will uniquely identify the state.
    ## An easy way is to form a tuple of representations of the components of
    ## the state, then form a string from that:
    def __repr__(self):
        return str((
            self.robot.__repr__(),
            [d.__repr__() for d in self.doors],
            [(room, sorted(contents))
            for room, contents in self.room_contents.items()]
        ))


In [5]:
ROOM_CONTENTS = {
    "workshop": {"rusty key"},
    "store room": {"bucket", "suitcase"},
    "tool cupboard": {"sledge hammer", "anvil", "saw", "screwdriver"},
    "charging room": {"charger"},
}

ITEM_WEIGHT = {
    "rusty key": 0,
    "bucket": 2,
    "suitcase": 4,
    "screwdriver": 1,
    "sledge hammer": 5,
    "anvil": 12,
    "saw": 2,
    "charger": 1000,
}

DOORS = [
    Door("workshop", "store room"),
    Door("store room", "tool cupboard", doorkey="rusty key", locked=True),
    Door("store room", "charging room")
]


In [6]:
from copy import deepcopy


class RobotWorker(SearchProblem):
    def __init__(self, state, goal_item_locations):
        self.initial_state = state
        self.goal_item_locations = goal_item_locations

    def possible_actions(self, state):

        robot_location = state.robot.location
        strength = state.robot.strength
        weight_carried = state.robot.weight_carried()
        battery_level = state.robot.battery_level

        actions = []

        if battery_level >= 1:
            # Can put down any carried item
            for i in state.robot.carried_items:
                actions.append(("put down", i))

            # Can pick up any item in room if strong enough
            for i in sorted(state.room_contents[robot_location]):
                if strength >= weight_carried + ITEM_WEIGHT[i]:
                    actions.append(("pick up", i))

            # If there is an unlocked door between robot location and
            # another location can move to that location
            for door in state.doors:
                if door.locked == False and robot_location in door.goes_between:
                    actions.append(("move to", door.other_loc[robot_location]))

            # If there is a locked door between robot location and another location robot can unlock the door if it has the required key
            for index, door in enumerate(state.doors):
                if (
                    door.locked == True
                    and robot_location in door.goes_between
                    and door.doorkey in state.robot.carried_items
                ):
                    actions.append(("unlock", index))

        if (
            battery_level < state.robot.MAX_BATTERY
            and robot_location == "charging room"
        ):
            actions.append(("charge", None))

        # Now the actions list should contain all possible actions
        return actions

    def cost(self, path, state):
        """
        This is an optional method that you only need to define if you are using
        a cost based algorithm such as "uniform cost" or "A*". It should return
        the cost of reaching a given state via a given path.
        If this is not defined, it will is assumed that each action costs one unit
        of effort to perform, so it returns the length of the path.
        """

        # Path is sequence of actions taken so far

        # Expensive move means search favours trips with max carrying capacity
        # Expensive charge means robot only charges when needed
        cost_lookup = {
            "put down": 1,
            "pick up": 1,
            "move to": 5,
            "unlock": 1,
            "charge": 10,
        }

        total_cost = 0
        for act, _ in path:
            total_cost += cost_lookup[act]

        return total_cost

    def heuristic(self, state):
        """
        This is an optional method that should return a heuristic value for any
        state. The value should be an estimate of the remaining cost that will be
        required to reach a goal. For an "admissible" heuristic, the value should
        always be equal to or less than the actual cost.
        """
        raise NotImplementedError

    def successor(self, state, action):
        next_state = deepcopy(state)
        act, target = action

        if act == "put down":
            next_state.robot.carried_items.remove(target)
            next_state.room_contents[state.robot.location].add(target)
            next_state.robot.battery_level -= 1

        if act == "pick up":
            next_state.robot.carried_items.append(target)
            next_state.room_contents[state.robot.location].remove(target)
            next_state.robot.battery_level -= 1

        if act == "move to":
            next_state.robot.location = target
            next_state.robot.battery_level -= 1

        if act == "unlock":
            next_state.doors[target].locked = False
            next_state.robot.battery_level -= 1

        if act == "charge":
            next_state.robot.battery_level = next_state.robot.MAX_BATTERY

        return next_state

    def goal_test(self, state):
        # print(state.room_contents)
        for room, contents in self.goal_item_locations.items():
            for i in contents:
                if not i in state.room_contents[room]:
                    return False
        return True

    def display_state(self, state):
        print("Robot location:", state.robot.location)
        print("Robot carrying:", state.robot.carried_items)
        print("Room contents:", state.room_contents)

In [7]:
rob = Robot('store room', [], 15, 10 )

state = State(rob, DOORS, ROOM_CONTENTS)

goal_item_locations =  {"store room":{"sledge hammer", "screwdriver", "anvil"}}

RW_PROBLEM_1 = RobotWorker( state, goal_item_locations )

In [8]:
poss_acts = RW_PROBLEM_1.possible_actions( RW_PROBLEM_1.initial_state )
poss_acts

[('pick up', 'bucket'),
 ('pick up', 'suitcase'),
 ('move to', 'workshop'),
 ('move to', 'charging room')]

In [9]:
for act in poss_acts:
    print("Action", act, "leads to the following state:")
    next_state = RW_PROBLEM_1.successor( RW_PROBLEM_1.initial_state, act )
    RW_PROBLEM_1.display_state(next_state)
    print()

Action ('pick up', 'bucket') leads to the following state:
Robot location: store room
Robot carrying: ['bucket']
Room contents: {'workshop': {'rusty key'}, 'store room': {'suitcase'}, 'tool cupboard': {'saw', 'screwdriver', 'anvil', 'sledge hammer'}, 'charging room': {'charger'}}

Action ('pick up', 'suitcase') leads to the following state:
Robot location: store room
Robot carrying: ['suitcase']
Room contents: {'workshop': {'rusty key'}, 'store room': {'bucket'}, 'tool cupboard': {'saw', 'screwdriver', 'anvil', 'sledge hammer'}, 'charging room': {'charger'}}

Action ('move to', 'workshop') leads to the following state:
Robot location: workshop
Robot carrying: []
Room contents: {'workshop': {'rusty key'}, 'store room': {'bucket', 'suitcase'}, 'tool cupboard': {'saw', 'screwdriver', 'anvil', 'sledge hammer'}, 'charging room': {'charger'}}

Action ('move to', 'charging room') leads to the following state:
Robot location: charging room
Robot carrying: []
Room contents: {'workshop': {'rusty

In [10]:
search( RW_PROBLEM_1, 'BF/FIFO', 10000000, loop_check=True)

This is the general SearchProblem parent class
You must extend this class to encode a particular search problem.

** Running Brandon's Search Algorithm **
Strategy: mode=BF/FIFO, cost=None, heuristic=None
Max search nodes: 10000000  (max number added to queue)
Searching (will output '.' each 1000 goal_tests)
.................................................................................................... (100000)
.......................................
:-)) *SUCCESS* ((-:

Path length = 17
Goal state is:
Robot location: store room
Robot carrying: ['rusty key']
Room contents: {'workshop': set(), 'store room': {'screwdriver', 'anvil', 'sledge hammer', 'bucket', 'suitcase'}, 'tool cupboard': {'saw'}, 'charging room': {'charger'}}
The action path to the solution is:
    ('move to', 'workshop')
    ('pick up', 'rusty key')
    ('move to', 'store room')
    ('unlock', 1)
    ('move to', 'tool cupboard')
    ('pick up', 'anvil')
    ('pick up', 'screwdriver')
    ('move to', 'store room')


'GOAL_STATE_FOUND'

In [11]:
# h1: relaxed_route: heuristic that ignores strength and battery limits
def relaxed_route(self, state):
    return self._estimate(state, with_limits=False)

# h2: route_plus_limits: heuristic that takes into account strength and battery limits
def route_plus_limits(self, state):
    return self._estimate(state, with_limits=True)

# Estimate the cost to reach the goal from a given state, optionally taking into account strength and battery limits.
def _estimate(self, state, with_limits):
    robot = state.robot
    carried = set(robot.carried_items)
    item_room = {item: room for room, items in state.room_contents.items() for item in items}
    undelivered = [item for item in self.goal_items
                   if item not in state.room_contents[self.goal_room]]
    # If there are no undelivered items, the cost to reach the goal is 0.
    if not undelivered:
        return 0
    pick_ups = sum(item not in carried for item in undelivered)
    put_downs = len(undelivered)
    must_visit = {self.goal_room} | {item_room[item] for item in undelivered if item not in carried}
    key_pick_ups, unlocks, moves = self._route(state, carried, item_room, undelivered,
                                               must_visit, with_limits)
    charges = 0
    # with_limits: check if the robot has enough battery to perform all the actions needed to reach the goal. 
    # If not, calculate how many charges are needed and include them in the cost.
    if with_limits:
        actions_needed = pick_ups + put_downs + key_pick_ups + unlocks + moves
        if actions_needed > robot.battery_level:
            if robot.battery_level < self.charger_distance[robot.location]:
                return DEAD_END
            charges = ceil((actions_needed - robot.battery_level) / robot.MAX_BATTERY)
            key_pick_ups, unlocks, moves = self._route(state, carried, item_room, undelivered,
                                                       must_visit | {CHARGING_ROOM}, with_limits)
    cost = self.action_cost
    # Return the estimated cost to reach the goal from the current state, 
    # taking into account the number of actions needed and their associated costs.
    return (cost["pick up"] * (pick_ups + key_pick_ups) + cost["put down"] * put_downs
            + cost["unlock"] * unlocks + cost["move to"] * moves + cost["charge"] * charges)

# Calculate the number of crossings required for a given door, 
# taking into account the robot's location, carried items, and undelivered items.
# If with_limits is True, also consider the robot's strength and calculate the number of loads needed to carry items beyond the door.
def _crossings(self, i, state, carried, item_room, undelivered, must_visit, with_limits):
    far_side = self.far_side[i]    # rooms beyond door i, away from the store room
    robot_on_far_side = state.robot.location in far_side
    if robot_on_far_side:
        crossings = 1
    else:
        crossings = 2 if any(room in far_side for room in must_visit) else 0
    if with_limits:
        weight_beyond = sum(ITEM_WEIGHT[item] for item in undelivered
                            if item_room.get(item) in far_side
                            or (item in carried and robot_on_far_side))
        loads = ceil(weight_beyond / state.robot.strength)
        crossings = max(crossings, 2 * loads - 1 if robot_on_far_side else 2 * loads)
    return crossings

In [12]:
from math import ceil

# Constants
CHARGING_ROOM = "charging room"
ACTION_TYPES = ("pick up", "put down", "move to", "unlock", "charge")
DEAD_END = float("inf")

class RobotWorkerHeuristics:
    """Two admissible heuristics for a RobotWorker problem.

    Both are based on the cost of the actions that every plan from a given state must still contain,
    ignoring the order in which they occur.

    h1: relaxed_route - cost still needed if the robot had unlimited strength and battery.
    h2: route_plus_limits - h1 plus the extra loads and charging forced by the strength and battery limits.
    """
    def __init__(self, problem):
        # store the problem's initial state and goal information
        start = problem.initial_state
        self.rooms = list(start.room_contents)

        # Goal: a single room and the items that must lie in it
        if len(problem.goal_item_locations) != 1:
            raise ValueError("The goal must name exactly one room")
        (self.goal_room, goal_items), = problem.goal_item_locations.items()
        self.goal_items = sorted(goal_items)

        # Action costs, read from the problem's own cost function
        self.action_cost = {action: problem.cost([(action, None)], start)
                            for action in ACTION_TYPES}

        # Doors: check they form a tree, then store the far side of each
        door_rooms = [tuple(door.goes_between) for door in start.doors]
        if (len(door_rooms) != len(self.rooms) - 1
                or self._reachable(self.rooms[0], door_rooms) != set(self.rooms)):
            raise ValueError("The doors must join the rooms as a tree")
        # If the charging room is not present in the list of rooms, raise an error
        # This is more of a sanity check than a requirement, since the charging room is not needed to reach the goal.
        if CHARGING_ROOM not in self.rooms:
            raise ValueError(f"No room called '{CHARGING_ROOM}'")
        # far_side[i]: rooms cut off from the goal room if door i were removed
        self.far_side = [set(self.rooms) - self._reachable(self.goal_room, door_rooms, skip_door=i)
                         for i in range(len(door_rooms))]
        # Fewest moves from each room to the charging room (locks ignored)
        self.charger_distance = {room: sum(self._separates(i, room, CHARGING_ROOM)
                                           for i in range(len(door_rooms)))
                                 for room in self.rooms}

    def relaxed_route(self, state):
        """h1: cost still needed if the robot had unlimited strength and battery.
        it would be as if the robot could carry all undelivered items in one load, and never need to recharge."""
        return self._estimate(state, with_limits=False)

    def route_plus_limits(self, state):
        """h2: h1 but instead of assuming unlimited strength and battery, it takes into account the robot's strength and battery limits.
        it would now be as if the robot could carry only loads no heavier than its strength, 
        and must recharge if it does not have enough battery to perform all the actions needed to reach the goal."""
        return self._estimate(state, with_limits=True)

    def _estimate(self, state, with_limits):
        """_estimate the cost to reach the goal from a given state, optionally taking into account strength and battery limits.
        this is what both heuristics have in common: they are based on the cost of the actions that every plan from a given state must still contain,"""
        robot = state.robot
        carried = set(robot.carried_items)
        item_room = {item: room for room, items in state.room_contents.items() for item in items}

        # Goal items not yet lying in the goal room: each needs a put down,
        # plus a pick up if the robot is not already carrying it
        undelivered = [item for item in self.goal_items
                       if item not in state.room_contents[self.goal_room]]
        if not undelivered:
            return 0
        pick_ups = sum(item not in carried for item in undelivered)
        put_downs = len(undelivered)

        # must_visit: rooms that the robot must reach to deliver all undelivered items
        must_visit = {self.goal_room} | {item_room[item] for item in undelivered if item not in carried}
        key_pick_ups, unlocks, moves = self._route(state, carried, item_room, undelivered,
                                                   must_visit, with_limits)

        charges = 0
        # if with_limits is True, check if the robot has enough battery to perform all the actions needed to reach the goal. 
        # If not, calculate how many charges are needed and include them in the cost.
        if with_limits:
            actions_needed = pick_ups + put_downs + key_pick_ups + unlocks + moves
            if actions_needed > robot.battery_level:
                if robot.battery_level < self.charger_distance[robot.location]:
                    # If the robot does not have enough battery to reach the charging room, 
                    # return DEAD_END to indicate that this state is not reachable.
                    return DEAD_END
                # Calculate the number of charges needed to reach the goal and include them in the cost.
                charges = ceil((actions_needed - robot.battery_level) / robot.MAX_BATTERY)
                key_pick_ups, unlocks, moves = self._route(state, carried, item_room, undelivered,
                                                           must_visit | {CHARGING_ROOM}, with_limits)

        cost = self.action_cost
        # Return the estimated cost to reach the goal from the current state, 
        # taking into account the number of actions needed and their associated costs.
        return (cost["pick up"] * (pick_ups + key_pick_ups)
                + cost["put down"] * put_downs
                + cost["unlock"] * unlocks
                + cost["move to"] * moves
                + cost["charge"] * charges)

    def _route(self, state, carried, item_room, undelivered, must_visit, with_limits):
        """takes a state and returns the number of keys to pick up, doors to unlock, and door crossings needed to reach `must_visit`."""
        location = state.robot.location
        must_visit = set(must_visit)
        doors_to_unlock, keys_to_fetch = set(), set()

        # Find all doors that must be unlocked to reach the rooms in `must_visit`.
        found_new_door = True
        # while there are still new doors to unlock, keep checking for locked doors that separate the robot from the rooms it must reach.
        while found_new_door:
            found_new_door = False
            for i, door in enumerate(state.doors):
                if (door.locked and i not in doors_to_unlock
                        and any(self._separates(i, location, room) for room in must_visit)):
                    doors_to_unlock.add(i)
                    found_new_door = True
                    if door.doorkey not in carried:
                        keys_to_fetch.add(door.doorkey)
                        if door.doorkey in item_room:
                            must_visit.add(item_room[door.doorkey])

        # Calculate the number of door crossings needed to reach all the rooms in `must_visit`.
        moves = sum(self._crossings(i, state, carried, item_room, undelivered, must_visit, with_limits)
                    for i in range(len(state.doors)))
        # return the number of keys to pick up, doors to unlock, and door crossings needed to reach `must_visit`.
        return len(keys_to_fetch), len(doors_to_unlock), moves

    def _crossings(self, i, state, carried, item_room, undelivered, must_visit, with_limits):
        """takes a door index and returns the fewest times the robot must still pass through door i.
        it does this by checking if the robot is on the far side of the door, and if any rooms that must be visited are on the far side.
        if with_limits is True, it also checks the weight of undelivered items on the far side,
        and calculates the number of loads needed to carry them, adjusting the number of crossings accordingly."""
        far_side = self.far_side[i]
        robot_on_far_side = state.robot.location in far_side

        # Determine the number of crossings needed based on the robot's location and the rooms that must be visited.
        if robot_on_far_side:
            crossings = 1
        else:
            crossings = 2 if any(room in far_side for room in must_visit) else 0

        # if with_limits is True, calculate the weight of undelivered items on the far side of the door,
        # and determine how many loads are needed to carry them, adjusting the number of crossings accordingly.
        if with_limits:
            weight_beyond = sum(ITEM_WEIGHT[item] for item in undelivered
                                if item_room.get(item) in far_side
                                or (item in carried and robot_on_far_side))
            loads = ceil(weight_beyond / state.robot.strength)
            crossings = max(crossings, 2 * loads - 1 if robot_on_far_side else 2 * loads)
        # return the calculated number of crossings needed for door i.
        return crossings

    # Check if a door separates two rooms.
    def _separates(self, i, room_a, room_b):
        return (room_a in self.far_side[i]) != (room_b in self.far_side[i])

    @staticmethod
    def _reachable(start_room, door_rooms, skip_door=None):
        """Check which rooms are reachable from start_room through the doors (locks ignored), optionally skipping one door.
        this is used to check if the doors form a tree, and to find the far side of each door."""
        reached, frontier = {start_room}, [start_room]
        # while there are still rooms to explore, keep checking for rooms that can be reached through the doors 
        # (ignoring locks), optionally skipping one door.
        while frontier:
            room = frontier.pop()
            for i, (room_a, room_b) in enumerate(door_rooms):
                if i != skip_door and room in (room_a, room_b):
                    other_room = room_b if room == room_a else room_a
                    if other_room not in reached:
                        reached.add(other_room)
                        frontier.append(other_room)
        # return the set of rooms that can be reached from start_room through the doors
        return reached

In [13]:
# Build both heuristics for RW_PROBLEM_1 and check their values at the initial state
robot_h = RobotWorkerHeuristics(RW_PROBLEM_1)
print("h1(initial state) =", robot_h.relaxed_route(RW_PROBLEM_1.initial_state))
print("h2(initial state) =", robot_h.route_plus_limits(RW_PROBLEM_1.initial_state))

# Run A* search using the second heuristic (h2) and return the search information.
astar_h2_search = search(RW_PROBLEM_1, 'BF/FIFO', 100000, loop_check=True,
                         cost=RW_PROBLEM_1.cost, heuristic=robot_h.route_plus_limits,
                         return_info=True)

h1(initial state) = 28
h2(initial state) = 58
This is the general SearchProblem parent class
You must extend this class to encode a particular search problem.

** Running Brandon's Search Algorithm **
Strategy: mode=BF/FIFO, cost=cost, heuristic=route_plus_limits
Max search nodes: 100000  (max number added to queue)
Searching (will output '.' each 1000 goal_tests)

:-)) *SUCCESS* ((-:

Path length = 17
Goal state is:
Robot location: store room
Robot carrying: ['rusty key']
Room contents: {'workshop': set(), 'store room': {'screwdriver', 'anvil', 'sledge hammer', 'bucket', 'suitcase'}, 'tool cupboard': {'saw'}, 'charging room': {'charger'}}
Cost of reaching goal: 58
The action path to the solution is:
    ('move to', 'workshop')
    ('pick up', 'rusty key')
    ('move to', 'store room')
    ('unlock', 1)
    ('move to', 'tool cupboard')
    ('pick up', 'anvil')
    ('pick up', 'screwdriver')
    ('move to', 'store room')
    ('put down', 'anvil')
    ('move to', 'charging room')
    ('c

In [14]:
# install psutil to monitor memory usage
# this is because one of the search tests crashes due to memory issues
%pip install psutil


[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
import gc
import random
import time
import pandas as pd
import psutil
from IPython.utils.capture import capture_output

# Constants for search tests
NODE_LIMIT = 500_000
# memory limit in GB for each search test; if exceeded, the test is stopped and reported as "Memory limit reached"
MEMORY_LIMIT_GB = 3
RANDOM_SEEDS = [1, 2, 3, 4, 5]
RESULTS_FILE = "robot_search_test_results.csv"

HEURISTICS = {"h1": "relaxed_route", "h2": "route_plus_limits"}

# The complete set of search tests:
SEARCH_TESTS = [
    ("BFS", None, "BF/FIFO", False, True, False, NODE_LIMIT),
    ("Uniform-cost", None, "BF/FIFO", True, True, False, NODE_LIMIT),
    ("DFS (fixed order)", None, "DF/LIFO", False, True, False, NODE_LIMIT),
    ("DFS (randomised)", None, "DF/LIFO", False, True, True, NODE_LIMIT),
    ("Greedy best-first", "h1", "BF/FIFO", False, True, False, NODE_LIMIT),
    ("Greedy best-first", "h2", "BF/FIFO", False, True, False, NODE_LIMIT),
    ("A*", "h1", "BF/FIFO", True, True, False, NODE_LIMIT),
    ("A*", "h2", "BF/FIFO", True, True, False, NODE_LIMIT),
    ("BFS", None, "BF/FIFO", False, False, False, NODE_LIMIT),
    ("DFS (fixed order)", None, "DF/LIFO", False, False, False, NODE_LIMIT),
    ("Greedy best-first", "h2", "BF/FIFO", False, False, False, NODE_LIMIT),
    ("A*", "h2", "BF/FIFO", True, False, False, NODE_LIMIT),
]

RESULT_LABELS = {
    "GOAL_STATE_FOUND": "Solved",
    "NODE_LIMIT_EXCEEDED": "Node limit reached",
    "SEARCH-SPACE_EXHAUSTED": "No solution",
    "MEMORY_LIMIT_REACHED": "Memory limit reached",
}

class MemoryLimitReached(Exception):
    """Raised when a search run uses more memory than MEMORY_LIMIT_GB."""

class MemoryLimitedProblem:
    """Wraps a search problem: counts nodes and stops the run if the kernel passes MEMORY_LIMIT_GB.
    Every other method and attribute is passed straight through to the wrapped problem."""

    def __init__(self, problem):
        self.problem = problem
        # Use psutil to monitor the memory usage of the current process
        self.process = psutil.Process()
        self.nodes_tested = 0
        self.nodes_generated = 1
        self.start_time = time.perf_counter()

    def __getattr__(self, name):
        return getattr(self.problem, name)

    def goal_test(self, state):
        self.nodes_tested += 1
        # if the number of nodes tested is a multiple of 1000 and the memory usage exceeds MEMORY_LIMIT_GB, raise MemoryLimitReached
        if self.nodes_tested % 1000 == 0 and self.process.memory_info().rss > MEMORY_LIMIT_GB * 1e9:
            raise MemoryLimitReached
        return self.problem.goal_test(state)

    def successor(self, state, action):
        self.nodes_generated += 1
        return self.problem.successor(state, action)

def run_search_test(problem, heuristics, heuristic_label, mode, use_cost, loop_check, randomise, node_limit, seed):
    # Run one bbqsearch test quietly and return its results, including the cost of the solution found.
    heuristic = getattr(heuristics, HEURISTICS[heuristic_label]) if heuristic_label else None
    limited_problem = MemoryLimitedProblem(problem)
    random.seed(seed)
    try:
        with capture_output():
            search_info = search(limited_problem, mode, node_limit,
                                 loop_check=loop_check, randomise=randomise,
                                 cost=problem.cost if use_cost else None, heuristic=heuristic,
                                 show_path=False, dots=False, return_info=True)
    except MemoryLimitReached:
        # The run is stopped without a solution; free its search queue before the next test
        gc.collect()
        return {
            "Result": "Memory limit reached",
            "Path length": None,
            "Solution cost": None,
            "Nodes tested": limited_problem.nodes_tested,
            "Nodes generated": limited_problem.nodes_generated,
            "Time (s)": round(time.perf_counter() - limited_problem.start_time, 2),
        }
    stats = search_info["search_stats"]
    path = search_info["result"]["path"]
    # Return a dictionary summarizing the results of the search test.
    return {
        "Result": RESULT_LABELS[search_info["result"]["termination_condition"]],
        "Path length": search_info["result"]["path_length"],
        "Solution cost": problem.cost(path, None) if path is not None else None,
        "Nodes tested": stats["nodes_tested"],
        "Nodes generated": stats["nodes_generated"],
        "Time (s)": round(stats["time_taken"], 2),
    }

In [16]:
# Run all search tests on the robot problem and save results to a CSV file
test_rows = []
for search_name, heuristic_label, mode, use_cost, loop_check, randomise, node_limit in SEARCH_TESTS:
    for seed in (RANDOM_SEEDS if randomise else [None]):
        results = run_search_test(RW_PROBLEM_1, robot_h, heuristic_label, mode, use_cost,
                                  loop_check, randomise, node_limit, seed)
        test_rows.append({
            "Search": search_name,
            "Heuristic": heuristic_label or "-",
            "Loop check": "On" if loop_check else "Off",
            "Seed": seed if seed is not None else "-",
            "Node limit": node_limit,
            **results,
        })
        # Save after every run so nothing is lost if the kernel stops
        pd.DataFrame(test_rows).to_csv(RESULTS_FILE, index=False)
        print(f"{search_name:18} {heuristic_label or '-':3} loop {'on ' if loop_check else 'off'} "
              f"seed {seed if seed is not None else '-':2}  {results['Result']:18} {results['Time (s)']:7.2f} s")
# Display the results in a DataFrame
test_results = pd.DataFrame(test_rows)
test_results["Path length"] = test_results["Path length"].astype("Int64")
test_results["Solution cost"] = test_results["Solution cost"].astype("Int64")
display(test_results)

BFS                -   loop on  seed -   Solved               27.06 s
Uniform-cost       -   loop on  seed -   Solved               35.86 s
DFS (fixed order)  -   loop on  seed -   Solved                2.15 s
DFS (randomised)   -   loop on  seed  1  Solved                0.05 s
DFS (randomised)   -   loop on  seed  2  Solved                0.04 s
DFS (randomised)   -   loop on  seed  3  Solved                0.07 s
DFS (randomised)   -   loop on  seed  4  Solved                0.33 s
DFS (randomised)   -   loop on  seed  5  Solved                0.78 s
Greedy best-first  h1  loop on  seed -   Solved                0.10 s
Greedy best-first  h2  loop on  seed -   Solved                0.01 s
A*                 h1  loop on  seed -   Solved                1.99 s
A*                 h2  loop on  seed -   Solved                0.03 s
BFS                -   loop off seed -   Node limit reached   28.37 s
DFS (fixed order)  -   loop off seed -   Memory limit reached   10.27 s
Greedy best-first 

,Search,Heuristic,Loop check,Seed,Node limit,Result,Path length,Solution cost,Nodes tested,Nodes generated,Time (s)
0,BFS,-,On,-,500000,Solved,17,58,139790,552846,27.06
1,Uniform-cost,-,On,-,500000,Solved,17,58,173208,642807,35.86
2,DFS (fixed order),-,On,-,500000,Solved,3256,16508,5826,23805,2.15
3,DFS (randomised),-,On,1,500000,Solved,214,741,364,1061,0.05
4,DFS (randomised),-,On,2,500000,Solved,192,659,314,965,0.04
5,DFS (randomised),-,On,3,500000,Solved,347,1265,523,1647,0.07
6,DFS (randomised),-,On,4,500000,Solved,704,2550,1050,3499,0.33
7,DFS (randomised),-,On,5,500000,Solved,2565,8853,3879,13374,0.78
8,Greedy best-first,h1,On,-,500000,Solved,17,58,703,2204,0.10
9,Greedy best-first,h2,On,-,500000,Solved,17,58,35,138,0.01


In [19]:
# Summarise the results of the search tests, combining repeated runs of the same configuration.
def describe(values, decimals=0):
    """One run: its value. Repeated runs: 'mean (min-max)'. No values: '-'."""
    values = [float(v) for v in values if pd.notna(v)]
    if not values:
        return "-"
    number = f"{{:,.{decimals}f}}"
    if len(values) == 1:
        return number.format(values[0])
    mean = sum(values) / len(values)
    return f"{number.format(mean)} ({number.format(min(values))}-{number.format(max(values))})"

# One row per search configuration (the 5 randomised DFS runs are combined)
summary_rows = []
for (search_name, heuristic_label, loop_check), runs in test_results.groupby(
        ["Search", "Heuristic", "Loop check"], sort=False):
    solved_runs = runs[runs["Result"] == "Solved"]
    summary_rows.append({
        "Search": search_name,
        "Heuristic": heuristic_label,
        "Loop check": loop_check,
        "Node limit": f"{runs['Node limit'].iloc[0]:,}",
        "Solved": f"{len(solved_runs)}/{len(runs)}",
        "Path length": describe(solved_runs["Path length"]),
        "Solution cost": describe(solved_runs["Solution cost"]),
        "Nodes tested": describe(runs["Nodes tested"]),
        "Nodes generated": describe(runs["Nodes generated"]),
        "Time (s)": describe(runs["Time (s)"], decimals=2),
    })
# Display the summary of the search test results
test_summary = pd.DataFrame(summary_rows)
display(test_summary)

,Search,Heuristic,Loop check,Node limit,Solved,Path length,Solution cost,Nodes tested,Nodes generated,Time (s)
0,BFS,-,On,"500,000",1/1,17,58,"139,790","552,846",27.06
1,Uniform-cost,-,On,"500,000",1/1,17,58,"173,208","642,807",35.86
2,DFS (fixed order),-,On,"500,000",1/1,"3,256","16,508","5,826","23,805",2.15
3,DFS (randomised),-,On,"500,000",5/5,"804 (192-2,565)","2,814 (659-8,853)","1,226 (314-3,879)","4,109 (965-13,374)",0.25 (0.04-0.78)
4,Greedy best-first,h1,On,"500,000",1/1,17,58,703,"2,204",0.10
5,Greedy best-first,h2,On,"500,000",1/1,17,58,35,138,0.01
6,A*,h1,On,"500,000",1/1,17,58,"9,745","40,750",1.99
7,A*,h2,On,"500,000",1/1,17,58,108,591,0.03
8,BFS,-,Off,"500,000",0/1,-,-,"186,336","500,001",28.37
9,DFS (fixed order),-,Off,"500,000",0/1,-,-,"22,000","51,332",10.27
